# Problem 1, (c)

In [21]:
import numpy as np
from fractions import Fraction

eta = 2.0**-1074
print(eta == np.nextafter(0.0, 1.0))

print(1.5 * eta == 2 * eta, 0.5 * eta == 0.0)

E = Fraction(1, 2**1074)
for x in [Fraction(3, 2) * E, Fraction(1, 2) * E]:
    fx = Fraction(float(x))
    print(abs(fx - x) / x)

True
True True
1/3
1


# Problem 2

In [22]:
import numpy as np

def myLU(A0, dtype=np.float64):
    A = np.array(A0, dtype=dtype)
    n = A.shape[0]
    for k in range(n - 1):
        A[k+1:, k] /= A[k, k]
        A[k+1:, k+1:] -= np.outer(A[k+1:, k], A[k, k+1:])
    return np.tril(A, -1) + np.eye(n, dtype=dtype), np.triu(A)

def myLUpivot(A0, dtype=np.float64):
    A = np.array(A0, dtype=dtype)
    n = A.shape[0]
    p = np.arange(n)
    for k in range(n - 1):
        m = np.argmax(np.abs(A[p[k:], k]))
        p[k], p[k+m] = p[k+m], p[k]
        A[p[k+1:], k] /= A[p[k], k]
        A[np.ix_(p[k+1:], range(k+1, n))] -= np.outer(A[p[k+1:], k], A[p[k], k+1:])
    Ap = A[p, :]
    return np.tril(Ap, -1) + np.eye(n, dtype=dtype), np.triu(Ap), p

def myL(L, b):
    y = b.copy()
    for k in range(1, len(b)):
        y[k] = b[k] - L[k, :k] @ y[:k]
    return y

def myU(U, y):
    x = y.copy()
    for k in range(len(y) - 1, -1, -1):
        x[k] = (y[k] - U[k, k+1:] @ x[k+1:]) / U[k, k]
    return x

In [23]:
A = np.array([[2, 4, -2], [4, 9, -3], [-2, -3, 7]], dtype=float)
b = np.array([2., 8., 10.])

L, U = myLU(A)
x = myU(U, myL(L, b))

Lp, Up, p = myLUpivot(A)
xp = myU(Up, myL(Lp, b[p]))

print(L, U, Lp, Up, p, sep="\n")

[[ 1.  0.  0.]
 [ 2.  1.  0.]
 [-1.  1.  1.]]
[[ 2.  4. -2.]
 [ 0.  1.  1.]
 [ 0.  0.  4.]]
[[ 1.          0.          0.        ]
 [-0.5         1.          0.        ]
 [ 0.5        -0.33333333  1.        ]]
[[ 4.          9.         -3.        ]
 [ 0.          1.5         5.5       ]
 [ 0.          0.          1.33333333]]
[1 2 0]


In [24]:
print(x, xp)
print(np.allclose(L @ U, A), np.allclose(Lp @ Up, A[p]), np.allclose(A @ xp, b))

[-1.  2.  2.] [-1.  2.  2.]
True True True


# Problem 3

In [25]:
def example(dtype, eps_):
    A = np.array([[eps_, 1], [1, 1]], dtype=dtype)
    b = np.array([1, 2], dtype=dtype)
    L, U = myLU(A, dtype);         x  = myU(U, myL(L, b))       # Ex. 6.4.3
    L, U, p = myLUpivot(A, dtype); xp = myU(U, myL(L, b[p]))    # Ex. 6.4.4
    print(f"{dtype.__name__:>8}  eps_=2^{int(np.log2(eps_))}  no pivot: {x}  pivot: {xp}")

for dtype in [np.float16, np.float32, np.float64]:
    t = int(-np.log2(np.finfo(dtype).eps))     # 10, 23, 52
    example(dtype, 2.0**-(t // 2))             # works
    example(dtype, 2.0**-(t + 3))              # collapses

 float16  eps_=2^-5  no pivot: [1.031 0.968]  pivot: [1.032 0.968]
 float16  eps_=2^-13  no pivot: [0. 1.]  pivot: [1. 1.]
 float32  eps_=2^-11  no pivot: [1.0004883 0.9995115]  pivot: [1.0004885 0.9995115]
 float32  eps_=2^-26  no pivot: [0. 1.]  pivot: [1. 1.]
 float64  eps_=2^-26  no pivot: [1.00000001 0.99999999]  pivot: [1.00000001 0.99999999]
 float64  eps_=2^-55  no pivot: [0. 1.]  pivot: [1. 1.]


# Problem 4

In [26]:
eta = 2.0**-1074
A = np.array([[1.75, 1.5, 0], [eta, 2*eta, 0], [0, 0, 1]])

print("Diagonally dominant:", all([abs(A[i, i]) > sum(abs(A[i, j]) for j in range(3) if j != i) for i in range(3)]))
with np.errstate(all="ignore"):
    L, U = myLU(A)
print(U)

Diagonally dominant: True
[[1.75 1.5  0.  ]
 [0.   0.   0.  ]
 [0.   0.    nan]]


In [29]:
np.isclose(1.0, 1.0 + 1e-12, atol=1e-12)

np.True_

In [46]:
np.isclose(1.0, [1.0 + 1e-13, 1.0 + 1e-14, 1.0 + 1e-12], atol=1e-13, rtol=0)


array([ True,  True, False])